In [ ]:
##EXPOSURE 2025 CLIMADA VERSION 5 

In [1]:
##Getting data infrastructure 
import geopandas as gpd
import pandas as pd
from pathlib import Path
import shapely
import os
import pickle
import numpy as np

from scipy.sparse import csr_matrix
from datetime import datetime
# on climada_python develop branch
from climada.util import coordinates as u_coords
from climada.entity.exposures.base import Exposures
from climada.entity.impact_funcs import ImpactFunc, ImpactFuncSet
from climada.engine import Impact
from climada.hazard.base import Hazard
from climada.util import lines_polys_handler as u_lp
from climada.util.api_client import Client
from climada_petals.entity.exposures import osm_dataloader as osm
from climada.util import coordinates as u_coords
from climada.hazard.base import Hazard
from climada import CONFIG
import urllib.request
from pathlib import Path
import logging
##############

import matplotlib.pyplot as plt
import shapely
import contextily as ctx
import os

import osm_flex
import osm_flex.download
import osm_flex.extract
import osm_flex.clip
import osm_flex.simplify

from climada import CONFIG
osm_flex.enable_logs()

In [2]:
iso3 = 'MOZ'
path_moz_dump = osm_flex.download.get_country_geofabrik(iso3)

INFO:osm_flex.download:Skip existing file: /cluster/home/gespejo/osm/osm_bpf/mozambique-latest.osm.pbf


In [3]:
# general paths & constants
PATH_DATA = '/cluster/work/climate/gespejo/data/infrastructure/'
#PATH_DATA_OSM = '/home/gespejogutierrez/programs/data/openstreetmap/' # path to search for osm.pbf files and to download to if not existing
PATH_DATA_OSM = '/cluster/home/gespejo/osm/osm_bpf/'# path to search for osm.pbf files and to download to if not existing
PATH_DATA_HVMV = PATH_DATA +'power_global/grid.gpkg' # path of this file from gridfinder. needs to be downloaded.
PATH_DATA_PP = PATH_DATA +'power_global/global_power_plant_database.csv' # path of this file from WRI global power plant db. needs to be downloaded.
#PATH_DATA_CT = PATH_DATA +'cell_towers/opencellid_global_1km_int.tif' # path of this file from worldbank open data gridded celltowers. needs to be downloaded.
PATH_DATA_POP = PATH_DATA + 'population/' # path to search for population files and to download to if not existing
PATH_SAVE = '/cluster/work/climate/gespejo/data/infrastructure/Mozambique_results_'

In [25]:
import os
from climada.util.coordinates import country_to_iso  # Ensure this function exists

# Define country and ISO3 code
cntry = 'Mozambique'
iso3 = country_to_iso(cntry)  # Converts 'Mozambique' → 'MOZ'

# Search for the correct OSM `.pbf` file dynamically
osm_files = [f for f in os.listdir(PATH_DATA_OSM) if f.endswith("-latest.osm.pbf")]

# Try to find the correct file based on the country name
path_osm_cntry = None
for filename in osm_files:
    if cntry.lower() in filename.lower():  # Match country name in filename
        path_osm_cntry = os.path.join(PATH_DATA_OSM, filename)
        break  # Stop searching after the first match

# If no file was found, raise an error
if not path_osm_cntry:
    raise FileNotFoundError(f"No OSM PBF file found for {cntry} in {PATH_DATA_OSM}")

# Construct paths for other datasets
path_worldpop_cntry_1km = os.path.join(PATH_DATA_POP, f"{iso3.lower()}_ppp_2020_1km_Aggregated_UNadj.tif")
path_worldpop_cntry_100m = os.path.join(PATH_DATA_POP, f"{iso3.lower()}_ppp_2020_UNadj.tif")
path_el_consump_cntry = os.path.join(PATH_DATA, f"power/Electricity consumption per capita - {cntry}.csv")
path_elgen_cntry = os.path.join(PATH_DATA, f"power/Electricity generation by source - {cntry}.csv")
path_el_imp_exp_cntry = os.path.join(PATH_DATA, f"power/Electricity imports vs. exports - {cntry}.csv")
path_save_cntry = PATH_SAVE + f'{iso3}/'
if not os.path.isdir(path_save_cntry):
    os.mkdir(path_save_cntry)

In [26]:
# COUNTRY SHAPE    
__, cntry_shape = u_coords.get_admin1_info([cntry])
cntry_shape = shapely.ops.unary_union([shp for shp in cntry_shape[iso3]])

In [27]:
# POWER LINES
gdf_powerlines_moz = gpd.read_file(PATH_DATA_HVMV, mask=cntry_shape)
gdf_powerlines_moz['osm_id'] = 'n/a'
gdf_powerlines_moz['ci_type'] = 'power_line' 
gdf_powerlines_moz = gdf_powerlines_moz[['osm_id', 'geometry', 'ci_type']]

In [30]:
gdf_powerlines_moz

,osm_id,geometry,ci_type
0,n/a,"LINESTRING (32.86042 -26.85625, 32.86875 -26.8...",power_line
1,n/a,"LINESTRING (32.85208 -26.85625, 32.84792 -26.8...",power_line
2,n/a,"LINESTRING (32.88542 -26.85208, 32.88125 -26.8...",power_line
3,n/a,"LINESTRING (32.87708 -26.83958, 32.87292 -26.8...",power_line
4,n/a,"LINESTRING (32.87708 -26.83958, 32.87708 -26.8...",power_line
...,...,...,...
4292,n/a,"LINESTRING (32.48542 -21.53542, 32.48958 -21.5...",power_line
4293,n/a,"LINESTRING (32.48542 -21.53542, 32.48125 -21.5...",power_line
4294,n/a,"LINESTRING (32.48125 -21.53125, 32.48542 -21.5...",power_line
4295,n/a,"LINESTRING (32.49375 -21.52708, 32.48958 -21.5...",power_line


In [31]:
## POWER PLANTS 
import pandas as pd
import geopandas as gpd

# Read the CSV file as a DataFrame
df_pp_world = pd.read_csv(PATH_DATA_PP)

# Convert it into a GeoDataFrame
gdf_pp_world = gpd.GeoDataFrame(
    df_pp_world, 
    geometry=gpd.points_from_xy(df_pp_world.longitude, df_pp_world.latitude), 
    crs="EPSG:4326")
gdf_pp_moz = gdf_pp_world[gdf_pp_world.country==f'{iso3}'][
    ['estimated_generation_gwh_2017','latitude','longitude',
     'name']]
del gdf_pp_world
if not gdf_pp_moz.empty:
    gdf_pp_moz = gpd.GeoDataFrame(
        gdf_pp_moz, geometry=gpd.points_from_xy(gdf_pp_moz.longitude, gdf_pp_moz.latitude))
else:
    gdf_pp_moz = CntryFileQuery.retrieve_cis('power')
    if len(gdf_pp_moz[gdf_pp_moz.power=='plant'])>1:
        gdf_pp_moz = gdf_pp_moz[gdf_pp_moz.power=='plant']
    else:
        # last 'resort': take generators frmom OSM
        gdf_pp_moz = gdf_pp_moz[gdf_pp_moz.power=='generator']
    gdf_pp_moz['geometry'] = gdf_pp_moz.geometry.apply(lambda geom: geom.centroid)
    gdf_pp_moz = gdf_pp_moz[['name', 'power', 'geometry']]
gdf_pp_moz['ci_type'] = 'power_plant'

/scratch/tmp.23127858.gespejo/ipykernel_341683/1411437849.py:6: DtypeWarning: Columns (10) have mixed types. Specify dtype option on import or set low_memory=False.
  df_pp_world = pd.read_csv(PATH_DATA_PP)


In [32]:
# Extract power infrastructure from the OSM dump
gdf_power_moz = osm_flex.extract.extract_cis(path_moz_dump, 'power')

# Define valid power types
power_types = ['generator', 'plant', 'substation', 'switch', 'transformer']

# Correct filtering using .isin()
gdf_power_moz = gdf_power_moz[gdf_power_moz['power'].isin(power_types)]

# Convert geometries to centroids of convex hulls
gdf_power_moz['geometry'] = gdf_power_moz.geometry.convex_hull.centroid

# Reset index
gdf_power_moz = gdf_power_moz.reset_index(drop=True)

INFO:osm_flex.extract:query is finished, lets start the loop
extract points: 100%|██████████| 31042/31042 [00:05<00:00, 5913.36it/s] 
INFO:osm_flex.extract:query is finished, lets start the loop
extract multipolygons: 100%|██████████| 29/29 [00:45<00:00,  1.58s/it]
INFO:osm_flex.extract:query is finished, lets start the loop
extract lines: 100%|██████████| 594/594 [00:07<00:00, 78.86it/s]
/scratch/tmp.23127858.gespejo/ipykernel_341683/174217427.py:11: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  gdf_power_moz['geometry'] = gdf_power_moz.geometry.convex_hull.centroid


In [33]:
gdf_power_moz

,osm_id,power,voltage,utility,name,geometry
0,912012527,substation,None,None,None,POINT (35.02126 -20.98745)
1,1902704592,transformer,None,None,None,POINT (32.38127 -25.89422)
2,2046156486,transformer,None,None,None,POINT (35.26376 -24.10582)
3,2390577870,transformer,None,None,None,POINT (32.48427 -25.93681)
4,2390577873,transformer,None,None,None,POINT (32.48442 -25.93689)
...,...,...,...,...,...,...
502,1332674957,generator,None,None,None,POINT (32.49304 -25.93493)
503,1332674958,generator,None,None,None,POINT (32.4931 -25.93491)
504,1332674959,generator,None,None,None,POINT (32.49338 -25.93482)
505,1332674960,substation,None,None,None,POINT (32.60542 -25.9504)


In [34]:
## hospitals 
gdf_health_moz = osm_flex.extract.extract_cis(path_moz_dump, 'healthcare')
gdf_health_moz['geometry'] = gdf_health_moz.geometry.apply(lambda geom: geom.centroid)
gdf_health_moz = gdf_health_moz[['name', 'geometry','amenity','building','healthcare']]
gdf_health_moz = gdf_health_moz[gdf_health_moz.geometry.within(cntry_shape)]
gdf_health_moz['ci_type'] = 'health'

INFO:osm_flex.extract:query is finished, lets start the loop
extract points: 100%|██████████| 798/798 [00:03<00:00, 251.07it/s] 
INFO:osm_flex.extract:query is finished, lets start the loop
extract multipolygons: 100%|██████████| 382/382 [00:37<00:00, 10.31it/s]


In [6]:
gdf_health_moz

NameError: name 'gdf_health_moz' is not defined

In [36]:
## educational facilities  
gdf_educ_moz = osm_flex.extract.extract_cis(path_moz_dump, 'education')
gdf_educ_moz['geometry'] = gdf_educ_moz.geometry.apply(lambda geom: geom.centroid)
gdf_educ_moz = gdf_educ_moz[['name', 'geometry','amenity','building']]
gdf_educ_moz = gdf_educ_moz[gdf_educ_moz.geometry.within(cntry_shape)]
gdf_educ_moz['ci_type'] = 'education'

INFO:osm_flex.extract:query is finished, lets start the loop
extract points: 100%|██████████| 294/294 [00:03<00:00, 87.49it/s] 
INFO:osm_flex.extract:query is finished, lets start the loop
extract multipolygons: 100%|██████████| 1488/1488 [00:43<00:00, 34.20it/s]


In [37]:
gdf_educ_moz

,name,geometry,amenity,building,ci_type
0,primary school,POINT (34.75546 -19.64221),school,None,education
2,SOS Children Village,POINT (35.4027 -23.88352),school,None,education
3,Escola Industrial e Comercial Estrela do Mar -...,POINT (35.19532 -21.5363),school,None,education
4,Escola Secundária de Chókwè,POINT (33.00343 -24.52927),school,None,education
5,Escola Básica de Chókwè,POINT (33.00439 -24.52947),school,None,education
...,...,...,...,...,...
1483,None,POINT (32.46551 -25.91435),None,school,education
1484,None,POINT (32.46541 -25.9144),None,school,education
1485,None,POINT (32.46531 -25.91445),None,school,education
1486,None,POINT (32.46517 -25.91461),None,school,education


In [38]:
# ROADS
# from osm; by default, take all types of roads
gdf_road_moz = osm_flex.extract.extract_cis(path_moz_dump, 'road') # for large countries: take main_road only and uncomment later ones
frac_unclass = gdf_road_moz.groupby('highway').size()['unclassified']/len(gdf_road_moz)

if frac_unclass < 0.2:
    # if unclassified road fraction is quite small, take only main roads
    gdf_road_moz = gdf_roads_moz[(gdf_road_moz.highway != 'residential') & 
                          (gdf_road_moz.highway != 'unclassified')]

gdf_road_moz = gdf_road_moz[gdf_road_moz.geometry.type=='LineString']
gdf_road_moz = gdf_road_moz[['osm_id','highway', 'geometry']]
gdf_road_moz = gdf_road_moz[gdf_road_moz.within(cntry_shape)]

INFO:osm_flex.extract:query is finished, lets start the loop
extract points: 0it [00:02, ?it/s]
INFO:osm_flex.extract:query is finished, lets start the loop
extract multipolygons: 100%|██████████| 4/4 [00:25<00:00,  6.46s/it]
INFO:osm_flex.extract:query is finished, lets start the loop
extract lines: 100%|██████████| 275175/275175 [00:35<00:00, 7696.45it/s] 


# TELECOM
# cells from rasterized opencellID (via WB)
path_ct_cntry = path_save_cntry+'celltowers.tif'
if not Path(path_ct_cntry).is_file():
    if cntry_shape.type=='Polygon':
        geo_mask = [cntry_shape]
    else:
        geo_mask = [mp for mp in cntry_shape]
    meta_ct, arr_ct = u_coords.read_raster(PATH_DATA_CT, src_crs={'epsg':'4326'},
                                           geometry=geo_mask)
    u_coords.write_raster(path_ct_cntry, arr_ct, meta_ct)
gdf_cells = nwu.load_resampled_raster(path_ct_cntry, 1/5)

In [39]:
# give ci_type attribute
gdf_pp_moz['ci_type'] = 'power_plant'
gdf_power_moz['ci_type'] = 'power_component'
gdf_powerlines_moz['ci_type'] = 'power_line'
gdf_health_moz['ci_type'] = 'health'
gdf_educ_moz['ci_type'] = 'education'
#gdf_cells['ci_type'] = 'celltower'
gdf_road_moz['ci_type'] = 'road'

In [40]:
# save locally to save time
for gdf in [gdf_pp_moz, gdf_power_moz, gdf_powerlines_moz, gdf_health_moz, gdf_educ_moz, gdf_road_moz]:
    gdf.to_feather(path_save_cntry+f'{gdf.ci_type.iloc[0]}')

In [ ]:
# general paths & constants
PATH_DATA = '/cluster/work/climate/gespejo/data/infrastructure/'
#PATH_DATA_OSM = '/home/gespejogutierrez/programs/data/openstreetmap/' # path to search for osm.pbf files and to download to if not existing
PATH_DATA_OSM = '/cluster/home/gespejo/osm/osm_bpf/'# path to search for osm.pbf files and to download to if not existing
PATH_DATA_HVMV = PATH_DATA +'power_global/grid.gpkg' # path of this file from gridfinder. needs to be downloaded.
PATH_DATA_PP = PATH_DATA +'power_global/global_power_plant_database.csv' # path of this file from WRI global power plant db. needs to be downloaded.
#PATH_DATA_CT = PATH_DATA +'cell_towers/opencellid_global_1km_int.tif' # path of this file from worldbank open data gridded celltowers. needs to be downloaded.
PATH_DATA_POP = PATH_DATA + 'population/' # path to search for population files and to download to if not existing
PATH_SAVE = '/cluster/work/climate/gespejo/data/infrastructure/Mozambique_results_'

In [5]:
import osm_flex
import osm_flex.download
import osm_flex.extract
cntry = 'Mozambique'
## new version using osm_flex
path_ctr_dump = Path(PATH_DATA_OSM+f'{cntry.lower()}-latest.osm.pbf')
if not Path(path_ctr_dump).exists():
    osm_flex.download.get_country_geofabrik(iso3,save_path=PATH_DATA_OSM)

# ROADS
# from osm; by default, take all types of roads
gdf_roads_orig = osm_flex.extract.extract_cis(path_ctr_dump, 'road')

INFO:osm_flex.extract:query is finished, lets start the loop
extract points: 0it [00:02, ?it/s]
INFO:osm_flex.extract:query is finished, lets start the loop
extract multipolygons: 100%|██████████| 4/4 [00:26<00:00,  6.53s/it]
INFO:osm_flex.extract:query is finished, lets start the loop
extract lines: 100%|██████████| 275175/275175 [00:25<00:00, 10947.08it/s]


In [ ]:
#backup

In [ ]:
import os
from climada.hazard.trop_cyclone import TropCyclone 
from datetime import datetime
#from climada.util.coordinates import _get_dist_to_coast_nasa_tif

# Set CLIMADA to use the local file this avoid to download the file that I downloaded manually and locally 
#os.environ["CLIMADA_DIST_TO_COAST_PATH"] = "/cluster/home/gespejo/climada/data/GMT_intermediate_coast_distance_01d.tif"

# Create an empty dictionary to store TropCyclone objects
tc_objects = {}

# Create dictionaries instead of lists to store TropCyclone names and ensemble variables
tc_names = {}
ens_variables = {}

for variable_name, forecast in forecast_variables.items():
    if f"{event_name}" in variable_name:
        ensemble_variable_name = f"ens_{variable_name}"
        ensemble_forecast = forecast.subset({'is_ensemble': True})
        ensemble_forecast.equal_timestep(time_step_h=0.5)
        globals()[variable_name] = forecast
        globals()[ensemble_variable_name] = ensemble_forecast

        print(f"Variable {ensemble_variable_name} created as the ensemble of {variable_name}")
        
        # Extract the date and time information from the variable name
        date_time_str = variable_name.split("_", 2)[-1]
        date_time_obj = datetime.strptime(date_time_str, "%Y%m%d%H%M%S")

        # Create a TropCyclone object from ensemble forecast data
        tc_object_name = f"{event_name}_{date_time_obj.strftime('%Y%m%d%H%M%S')}"
        
        # Assign name and ensemble variable to respective keys
        timestamp_key = f"{event_name}_{date_time_obj.strftime('%Y%m%d%H%M%S')}"
        tc_names[timestamp_key] = TropCyclone.from_tracks(ensemble_forecast, centroids=cent)

        # Assign ensemble variable name to the dict
        ens_variables[f"ens_{timestamp_key}"] = ensemble_forecast
        print(f"Variable ens_{timestamp_key} added to ens_variables")

In [ ]:
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import numpy as np
import datetime
import os

people_disrupt_nodisrupt_cat_folder = '/cluster/home/gespejo/MOZ_struc_plots_Idai'

# Loop through the keys in disrupt_pp1_unique_dict
for key in disrupt_pp1_unique_dict.keys():
    # Extract the date part from the key (e.g., '20190310120000')
    date_key = key.split('_')[-1]

    # Construct the corresponding keys for the other dictionaries
    key_pp2 = f'impp2_MOZ_Idai_{date_key}'
    key_imph2 = f'imph2_MOZ_new_Idai_{date_key}'
    key_imppeople = f'impp1_MOZ_Idai_{date_key}'  # For imppeople_nodisrupt_cat1_copy

    # Check if the keys exist in the respective dictionaries
    if key_pp2 in disrupt_pp2 and key_imph2 in disrupt_imph2 and key_imppeople in imppeople_nodisrupt_cat1_copy:
        # Retrieve the GeoDataFrames for the specified day
        disrupt_pp1_unique_gdf = disrupt_pp1_unique_dict[key]
        disrupt_pp2_df = disrupt_pp2[key_pp2]
        disrupt_imph2_df = disrupt_imph2[key_imph2]
        imppeople_nodisrupt_cat1_copy_df = imppeople_nodisrupt_cat1_copy[key_imppeople]

        # Parse the date string into a datetime object
        parsed_date = datetime.datetime.strptime(date_key, '%Y%m%d%H%M%S')

        # Format the date into the desired format: "forecast date: Thu 15 March 2019 12 UTC"
        formatted_date = parsed_date.strftime('forecast date: %a %d %B %Y %H UTC')

        # Create custom colormaps
        greens_cmap = plt.cm.Greens
        oranges_cmap = plt.cm.Oranges
        purples_cmap = plt.cm.Purples  # Purple replaces the gray for a better contrast

        # Adjust the colormaps to start with a darker shade for more distinction
        greens_colors = greens_cmap(np.linspace(0.2, 1, 256))  # Darker green starting at 0.2
        oranges_colors = oranges_cmap(np.linspace(0.2, 1, 256))  # Darker orange starting at 0.2
        purples_colors = purples_cmap(np.linspace(0.2, 1, 256))  # Darker purple starting at 0.2

        # Create new colormaps with adjusted starting shades
        new_greens_cmap = mcolors.ListedColormap(greens_colors)
        new_oranges_cmap = mcolors.ListedColormap(oranges_colors)
        new_purples_cmap = mcolors.ListedColormap(purples_colors)

        # Plot all layers on the map
        fig, ax = plt.subplots(figsize=(10, 10))

        # Plot the Mozambique boundary (gadm_MOZ_gdf) first as the base layer
        gadm_MOZ_gdf.plot(ax=ax, color='white', edgecolor='black')

        # Plot disrupt_pp1_unique points in green (Category 1 Unique) with the adjusted colormap
        disrupt_pp1_unique_gdf.plot(ax=ax, column='imp_people', cmap=new_greens_cmap, legend=False, edgecolor='none', markersize=40)

        # Plot disrupt_pp2 points in orange (Category 2) with the adjusted colormap
        disrupt_pp2_df.plot(ax=ax, column='imp_people', cmap=new_oranges_cmap, legend=False, edgecolor='none', markersize=40)

        # Plot imp_people_cat1_unique points in purple (Category 1 People Affected) for better contrast
        imppeople_nodisrupt_cat1_copy_df.plot(ax=ax, column='imp_people', cmap=new_purples_cmap, legend=False, edgecolor='none', markersize=40)

        # Define marker shapes for different levels in disrupt_imph2
        level_markers = {1: 'o', 2: 's', 3: '^', 4: 'D'}

        # Define a dictionary to map levels to their alpha, edgecolor, and facecolor settings
        level_style = {
            1: {'alpha': 0.7, 'edgecolor': 'indigo', 'facecolor': 'indigo', 's': 10},
            2: {'alpha': 0.7, 'edgecolor': 'mediumslateblue', 'facecolor': 'mediumslateblue', 's': 10},
            3: {'alpha': 0.7, 'edgecolor': 'darkmagenta', 'facecolor': 'darkmagenta', 's': 10},
            4: {'alpha': 0.7, 'edgecolor': 'mediumvioletred', 'facecolor': 'mediumvioletred', 's': 10},
        }

        # Plot disrupt_imph2 points by levels
        for level in disrupt_imph2_df['levels'].unique():
            level_df = disrupt_imph2_df[disrupt_imph2_df['levels'] == level]

            ax.scatter(
                level_df['longitude'], level_df['latitude'],
                marker=level_markers[level],
                label=f'Health Facility Level {level}',
                **level_style[level]
            )

        # Set the latitude and longitude limits
        ax.set_xlim([32, 36.5])  # Longitude range
        ax.set_ylim([-22.5, -17.5])  # Latitude range

        # Set the formatted date as the title of the plot
        ax.set_title(formatted_date, fontsize=15)

        # Add horizontal colorbars below the plot for each layer
        sm1 = plt.cm.ScalarMappable(cmap=new_greens_cmap, norm=plt.Normalize(vmin=disrupt_pp1_unique_gdf['imp_people'].min(), vmax=disrupt_pp1_unique_gdf['imp_people'].max()))
        sm2 = plt.cm.ScalarMappable(cmap=new_oranges_cmap, norm=plt.Normalize(vmin=disrupt_pp2_df['imp_people'].min(), vmax=disrupt_pp2_df['imp_people'].max()))
        sm3 = plt.cm.ScalarMappable(cmap=new_purples_cmap, norm=plt.Normalize(vmin=imppeople_nodisrupt_cat1_copy_df['imp_people'].min(), vmax=imppeople_nodisrupt_cat1_copy_df['imp_people'].max()))

        # Create horizontal colorbars with closer spacing
        cbar1 = fig.colorbar(sm1, ax=ax, orientation='horizontal', fraction=0.02, pad=0.07)  # Very close to the plot
        cbar1.set_label('People disrupted cat.1-above the threshold') 

        cbar2 = fig.colorbar(sm2, ax=ax, orientation='horizontal', fraction=0.02, pad=0.07)  # Very close to the first colorbar
        cbar2.set_label('People disrupted cat.2-above the threshold')

        cbar3 = fig.colorbar(sm3, ax=ax, orientation='horizontal', fraction=0.02, pad=0.05)  # Very close to the second colorbar
        cbar3.set_label('People direct affected')
        
        # Save the plot to a file in the output folder
        output_path = os.path.join(people_disrupt_nodisrupt_cat_folder, f"people_disrupt_nodisrupt_by_cat_{key}.jpg")
        plt.savefig(output_path, bbox_inches='tight', format='jpg')

        # Close the plot to free memory
        plt.close()

    else:
        print(f"Key {key} or its corresponding versions are missing in one of the dictionaries, skipping this entry.")